# Beat 2: Saliency Maps

**Pergunta:** E se eu derivar a saída em relação à imagem?

Este notebook demonstra o gradiente cru (3x224x224), a redução para um canal via `max|·|`, e a aplicação essencial da escala de exibição (`scale_for_display(high=0.99, gamma=0.5)`).

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('notebooks'))
import deck
import torch
import matplotlib.pyplot as plt
from PIL import Image
from gradcam.models import load_model, preprocess
from gradcam.saliency import saliency
from gradcam.viz import scale_for_display, gray_rgb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("vgg16", device=device)
model.eval()
print("Modelo VGG-16 carregado.")

## Calculando e exibindo o Saliency Map real

Carregando a imagem do gato/cachorro e calculando o gradiente em relação à classe prevista.

In [ ]:
base_dir = os.path.abspath('.')
if os.path.basename(base_dir) == 'notebooks':
    img_path = "assets/cat_dog.jpg"
else:
    img_path = "notebooks/assets/cat_dog.jpg"

pil_img = Image.open(img_path).convert("RGB")
tensor_img = preprocess(pil_img).to(device)

smap, class_idx = saliency(model, tensor_img, class_idx=None, mode="vanilla")
smap_scaled = scale_for_display(smap, high=deck.DISPLAY_HIGH, gamma=deck.DISPLAY_GAMMA)

fig, ax = plt.subplots(figsize=(5, 5), dpi=110)
ax.imshow(gray_rgb(smap_scaled))
ax.axis("off")
ax.set_title(f"Saliency Map (Classe ID: {class_idx})")
deck.show(fig)